# DIY Copilot
## Lesson: Basic RAG

This lesson is intended to show you the basics of a Retrieval Augmented Generation (RAG) system. With RAG, a coding assistant finds the relevant parts of your internal knowledge – business rules, engineering guidelines, security requirements – before it answers or writes code. This is how you teach an assistant the things that are *not* in the code.

During this lesson you will learn how to ...

- implement the different building blocks of RAG
- create an ingestion pipeline for a knowledge base from the building blocks
- create a retrieval pipeline from the building blocks
- use a RAG system to answer questions and to generate code that follows your company's rules

### Set up the environment

In [ ]:
# >>> SETUP <<<
# In Google Colab, this cell
#   1. clones the workshop repository,
#   2. installs its dependencies into the Python environment of this notebook (with uv),
#   3. makes the shared code (lib) importable.
# Locally (after `uv sync` in the repository), only step 3 is needed.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/openknowledge/diy-copilot-bootcamp"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/diy-copilot")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, str(REPO_DIR / "setup" / "install_dependencies.py")], check=True)
else:
    # The repository root is the first parent directory containing the shared code
    REPO_DIR = next(p for p in Path.cwd().resolve().parents if (p / "lib").is_dir() and (p / "pyproject.toml").exists())

# The repository is not an installed package, so we add it to the module search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

Workshop repository: /home/tim/workshops/diy-copilot


In [2]:
import uuid
from pathlib import Path

from chromadb import EphemeralClient
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from litellm import embedding

from lib.completion import generate_completion
from lib.constants import CODEBASE_PATH, KNOWLEDGE_BASE_PATH
from lib.environment import load_environment
from lib.rag import list_knowledge_base_files
from lib.retry import call_with_retry

In [3]:
# Models
default_models = load_environment()
generation_model = default_models["generation_model"]
embedding_model = default_models["embedding_model"]
print(f"Generation model: {generation_model}")
print(f"Embedding model:  {embedding_model}")

# Generation parameters
TEMPERATURE = 0.7
MAX_OUTPUT_TOKENS = 1000

# Retrieval parameters
TOP_K = 3
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# Prompts
SYSTEM_PROMPT = "You are a helpful coding assistant for the developers of our company."

Generation model: gemini/gemini-3.1-flash-lite
Embedding model:  gemini/gemini-embedding-001


## Generate a response without internal knowledge

In [4]:
# Let's ask a specific question about the rules of our company.
user_prompt = "At our company, what happens if nobody checks in to a meeting room booking?"
# The answer should mention that the booking is released 10 minutes after the start.

print(generate_completion(user_prompt=user_prompt, system_prompt=SYSTEM_PROMPT, model_name=generation_model))

To provide you with the most accurate information, I would need to know which internal system or office location you are referring to. However, **most modern office booking systems** (such as Microsoft Outlook/Exchange, Google Workspace, or dedicated platforms like Envoy or Robin) typically follow a standard "no-show" workflow:

### 1. The "Auto-Release" Policy
In most offices, if a meeting room is booked but the organizer does not manually check in within a specific window (usually **5 to 15 minutes** after the meeting start time), the system will trigger an **Auto-Release**.

### 2. What happens during an Auto-Release:
*   **Booking Cancellation:** The reservation is automatically removed from the room’s calendar.
*   **Room Availability:** The room is marked as "Available" or "Free" on the scheduling display (the tablet outside the door) and in the digital booking system.
*   **Notification:** The meeting organizer often receives an automated email or notification stating that the r

## Define helpers

In [5]:
# This will be the chromadb collection we use as a knowledge base. The EphemeralClient keeps everything in memory.
chromadb_collection = EphemeralClient().get_or_create_collection(name="default")


def peek_knowledgebase(n: int = 3):
    """Shows the first n items of the knowledge base."""
    items = chromadb_collection.peek(limit=n)
    for document, metadata in zip(items["documents"], items["metadatas"]):
        print(f"----- {metadata} -----\n{document}\n")

### Define RAG Building Blocks

In [6]:
# Building Block "Loading": Get the content of a document of the knowledge base.
def load_file_content(file_name: str) -> str:
    """Loads the content of a document of the knowledge base.

    Parameters
    ----------
    file_name : str
        The path of the document relative to the root of the knowledge base
    Returns
    -------
    file_content : str
        The content of the file as a string
    """
    return (KNOWLEDGE_BASE_PATH / file_name).read_text(encoding="utf-8")

#### Chunking structured documents
A plain text splitter cuts wherever the chunk is full – possibly in the middle of a section. LangChain's `RecursiveCharacterTextSplitter.from_language()` knows the syntax of Markdown (and of many programming languages) and prefers to split at headings, so that a chunk tends to contain a complete section. Let's compare both.

In [7]:
# Building Block "Chunking": Split the content into smaller chunks
def do_chunk(text: str, file_name: str) -> list[str]:
    """Chunks a text with a splitter that matches the type of the file.

    Parameters
    ----------
    text : str
        The text to be chunked
    file_name : str
        The name of the file; its extension determines the splitter
    Returns
    -------
    chunks: list[str]
        The created chunks
    """
    language = {".py": Language.PYTHON, ".md": Language.MARKDOWN}.get(Path(file_name).suffix)
    if language is None:
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    else:
        text_splitter = RecursiveCharacterTextSplitter.from_language(
            language=language, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
    return text_splitter.split_text(text)

In [8]:
# Compare a plain splitter with the Markdown-aware splitter on the booking policy
sample = load_file_content("domain/room-booking-policy.md")
plain_chunks = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP).split_text(sample)
markdown_chunks = do_chunk(sample, "domain/room-booking-policy.md")

print("First lines of each chunk (plain splitter):")
for chunk in plain_chunks:
    print("   ", chunk.splitlines()[0])
print("\nFirst lines of each chunk (Markdown splitter):")
for chunk in markdown_chunks:
    print("   ", chunk.splitlines()[0])

First lines of each chunk (plain splitter):
    # Meeting Room Booking Policy
    ### 2.2 Duration
    The number of attendees must not exceed the number of seats of the room.
    ## 4. Check-in and no-shows
    ## 5. Cancellation

First lines of each chunk (Markdown splitter):
    # Meeting Room Booking Policy
    ### 2.2 Duration
    ### 3.1 Capacity
    ## 4. Check-in and no-shows
    ## 5. Cancellation


In [9]:
# Building Block "Embedding": Create multi-dimensional embeddings for a given chunk.
def do_embed(chunk: str) -> list[float]:
    """Embeds a given chunk and returns the embedding.

    Parameters
    ----------
    chunk : str
        The chunk to be embedded
    Returns
    -------
    embedding: list[float]
        The created embedding
    """
    # call_with_retry waits and retries if the free tier rate limit is reached
    response = call_with_retry(embedding, model=embedding_model, input=chunk)
    return response["data"][0]["embedding"]


def do_batch_embed(chunks: list[str], batch_size: int = 100) -> list[list[float]]:
    """Embeds a list of chunks in batches and returns the embeddings.

    Sending many chunks per request is much faster and saves requests - important
    for the rate limits of the free tier.

    Parameters
    ----------
    chunks : list[str]
        The chunks to be embedded
    batch_size : int
        The number of chunks per request (max. 100 for Gemini)
    Returns
    -------
    embeddings: list[list[float]]
        The created embeddings
    """
    all_embeddings = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        response = call_with_retry(embedding, model=embedding_model, input=batch)
        all_embeddings.extend(data["embedding"] for data in response["data"])
    return all_embeddings

In [10]:
# Building Block "Knowledge base": Store embeddings, chunks and metadata in a vector store
def persist_embeddings(chunks: list[str], embeddings: list[list[float]], metadatas: list[dict]) -> None:
    """Persists the chunks, their embeddings and metadata in the knowledge base.

    Parameters
    ----------
    chunks : list[str]
        The chunks to be persisted
    embeddings: list[list[float]]
        The corresponding embeddings to be persisted
    metadatas: list[dict]
        The corresponding metadata, e.g. the document the chunk comes from
    """
    ids = [str(uuid.uuid4()) for _ in chunks]
    chromadb_collection.add(ids=ids, documents=chunks, embeddings=embeddings, metadatas=metadatas)

In [11]:
# Building Block "Augmentation": Create an updated prompt by merging the original user input with the provided context
def augment(user_input: str, context: list[str]) -> str:
    """Augments a given user input by merging it with the provided context.

    Parameters
    ----------
    user_input : str
        The user input to be augmented
    context : list[str]
        The context to be merged with the user input
    Returns
    -------
    augmented_prompt: str
        The created augmented prompt
    """
    prepared_context = "\n\n---\n\n".join(context)
    return f"""Answer the question as precisely as possible, using only the provided
context from our internal knowledge base. Mention the documents you used. If the answer
is not in the context, say "The answer is not available in the context." and do
not make anything up.

Context:
{prepared_context}

Question:
{user_input}

Answer:
"""

In [12]:
# Building Block "Top-k Fetching": Get the k semantically closest chunks to the user input from the knowledge base
def do_top_k_fetching(user_input_embedding: list[float], top_k: int) -> list[str]:
    """Fetches the k semantically closest chunks to the user input from the knowledge base.

    Parameters
    ----------
    user_input_embedding : list[float]
        The embedding of the user input
    top_k : int
        The number of semantically closest chunks to be fetched

    Returns
    -------
    context: list[str]
        The fetched chunks, each prefixed with the document it comes from
    """
    # Chroma allows querying for multiple embeddings at once. We query for one,
    # so we only use the first list of results.
    result = chromadb_collection.query(query_embeddings=[user_input_embedding], n_results=top_k)
    return [
        f"Document: {metadata['source']}\n{document}"
        for document, metadata in zip(result["documents"][0], result["metadatas"][0])
    ]

In [13]:
# Building Block "Generation": Use the generation model to create a response
def generate_response(prompt: str) -> None:
    """Generates a response for a given prompt and prints it.

    Parameters
    ----------
    prompt : str
        The prompt to be used for the generation
    """
    response = generate_completion(
        model_name=generation_model,
        system_prompt=SYSTEM_PROMPT,
        user_prompt=prompt,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )
    print(response)

### Create the ingestion pipeline

In [14]:
def do_ingestion(file_names: list[str]) -> None:
    """Ingests a list of documents of the knowledge base.

    Parameters
    ----------
    file_names : list[str]
        The paths of the documents to be ingested, relative to the root of the knowledge base
    """
    all_chunks, all_metadatas = [], []
    for file_name in file_names:
        # Load the document content
        file_content = load_file_content(file_name)

        # Chunk the content into smaller chunks
        chunks = do_chunk(file_content, file_name)
        all_chunks.extend(chunks)
        all_metadatas.extend({"source": file_name} for _ in chunks)

    # Embed the chunks of all files together to save requests
    embeddings = do_batch_embed(all_chunks)

    # Persist the chunks, embeddings and metadata in the knowledge base
    persist_embeddings(all_chunks, embeddings, all_metadatas)
    print(f"Ingested {len(all_chunks)} chunks from {len(file_names)} documents.")

### Perform ingestion

In [15]:
# Ingest all documents of the knowledge base: domain, engineering and security
file_names = list_knowledge_base_files(KNOWLEDGE_BASE_PATH)
do_ingestion(file_names)

Ingested 36 chunks from 11 documents.


In [16]:
# Use the helper function to peek into the knowledge base
peek_knowledgebase()

----- {'source': 'domain/desk-sharing-policy.md'} -----
# Desk Sharing Policy

- Owner: Facility Management
- Applies to: all employees working in the Berlin office
- Version: 1.4

Since 2024 the Berlin office works with shared desks. Desks are booked with
the desk booking app "Deskly" (not with Roomie, which is only used for meeting
rooms).

## Booking a desk

- Desks can be booked at most **14 days in advance**.
- A desk is always booked for a whole day; half-day bookings are not possible.
- Every employee can hold at most **3 desk bookings** in the future at the same time.
- Team areas: each team has a home area where its members should book first.

## Cancellation

Desk bookings can be cancelled until **10:00 on the same day**. Later
cancellations are counted as a no-show.

## Check-in

Desks have to be checked in by scanning the QR code on the desk until
**10:30**. Desks that are not checked in are released at 10:30 and can be
booked by others for the rest of the day.

----- {'sou

### Exercise 01: Create the RAG pipeline
In this exercise you will create a RAG pipeline for retrieving relevant chunks and generating a grounded response.

In [17]:
def do_rag(user_input: str, verbose: bool = False) -> None:
    """Runs the RAG pipeline with a given user input and prints the response.

    Parameters
    ----------
    user_input : str
        The user input to be used for the RAG pipeline
    verbose : bool, optional [default: False]
        Whether to print details of the RAG process or not
    """
    # TODO: Embed the user input
    user_input_embedding = do_embed(chunk=user_input)

    # TODO: "R" like "Retrieval": Get the k semantically closest chunks to the user input from the knowledge base
    context = do_top_k_fetching(user_input_embedding=user_input_embedding, top_k=TOP_K)
    if verbose:
        print("Retrieved context:")
        for chunk in context:
            print(f"----------\n{chunk}")
        print("----------\n")

    # TODO: "A" like "Augmented": Create the augmented prompt
    augmented_prompt = augment(user_input=user_input, context=context)

    # TODO: "G" like "Generation": Generate a response
    generate_response(prompt=augmented_prompt)

In [18]:
# Define the user input. This should be a question about our internal knowledge.
user_input = "At our company, what happens if nobody checks in to a meeting room booking?"
# The answer should mention that the booking is released 10 minutes after the start.

# TODO: Perform RAG
do_rag(user_input=user_input, verbose=True)

Retrieved context:
----------
Document: domain/room-booking-policy.md
## 4. Check-in and no-shows

### 4.1 Check-in

Every booking must be checked in by the person who booked it. Check-in is
possible from **5 minutes before** until **10 minutes after** the start of
the meeting, on the display next to the door or with the Slack command
`/room checkin`.

### 4.2 Automatic release

If nobody checks in within 10 minutes after the start, the booking is
**released automatically** and the room is free for others. The person who
booked it receives an email, and the facility team is informed. A released
booking cannot be checked in anymore – book the room again if it is still
free.

This rule also applies to bookings of facility admins.

> Background: before the introduction of the check-in, about 30% of all
> booked meetings never took place while other colleagues searched for a room.
----------
Document: domain/glossary.md
| Term | Meaning |
|---|---|
| **Booking** | Reservation of a meeting 

### Exercise 02: Ask your own questions
Ask the RAG system some questions of your own – about the business rules, the engineering guidelines or the security requirements. Also try a question whose answer is **not** in the knowledge base. Does the assistant admit that it doesn't know?

In [19]:
# TODO: Ask your own questions
do_rag(user_input="How do I name a branch for a bug fix?")
print("==========")
do_rag(user_input="May I log the title of a meeting when a booking is created?")
print("==========")
do_rag(user_input="Can I bring my dog to the office?")

Bug fix branches should be named using the format `fix/<ticket>-short-name` (e.g., `fix/WORK-412-naive-datetimes`).

Source: `engineering/git-and-code-review.md`
Yes, you may log the title of a meeting, but you must adhere to the retention period specified in the documentation. According to the document **security/data-protection.md** (section DP-03), records of bookings (including the title and owner) must be anonymised after 12 months. Anonymisation requires removing the owner and replacing the title with a generic text. 

Additionally, per section DP-06, you must not paste booking data (including meeting titles) into public AI tools.
The answer is not available in the context.


### (Optional) Exercise 03: RAG for code generation
So far, we used RAG to answer questions. But a coding assistant mostly **writes code** – and this code must follow the business rules and guidelines of the company. Let's combine both kinds of context:
* the **code** the assistant works on (here: the relevant files of Roomie, provided directly – on day 2 the assistant will find them itself using tools)
* the **internal knowledge** retrieved from the knowledge base (business rules, guidelines)

Our task for the assistant: *"Add a method `extend_booking` to the `BookingService` that extends a booking by a given number of minutes."* Without context, a model would happily write `datetime.now()`, raise `HTTPException` and allow any duration. With the right context, it should use the injected clock, raise a domain error and respect the maximum duration of 4 hours.

Your task is to implement `generate_code_with_rag`:
1. Retrieve the relevant chunks from the knowledge base for the task.
2. Build a prompt containing the task, the retrieved knowledge and the code files.
3. Generate the code.

In [20]:
CODE_FILES = ["roomie/services/booking_service.py", "roomie/config.py", "roomie/errors.py"]


def load_code_files(code_files: list[str]) -> str:
    """Loads the given files of the Roomie code base, each prefixed with its path."""
    return "\n\n".join(
        f"### File: {path}\n{(CODEBASE_PATH / path).read_text(encoding='utf-8')}" for path in code_files
    )


def generate_code_with_rag(task: str, code_files: list[str], top_k: int = 5, verbose: bool = False) -> str:
    """Generates code for a task using retrieved internal knowledge and the given code files.

    Parameters
    ----------
    task : str
        The coding task
    code_files : list[str]
        Paths of the relevant code files, relative to the root of the code base
    top_k : int
        Number of chunks to retrieve from the knowledge base
    verbose : bool
        Whether to print the documents the knowledge was retrieved from
    Returns
    -------
    str
        The generated code
    """
    # TODO: Retrieve the relevant knowledge for the task
    knowledge = do_top_k_fetching(user_input_embedding=do_embed(task), top_k=top_k)
    if verbose:
        print("Retrieved knowledge from:", [chunk.splitlines()[0] for chunk in knowledge], "\n")

    # TODO: Build the prompt from the task, the knowledge and the code
    prepared_knowledge = "\n\n---\n\n".join(knowledge)
    prompt = f"""You are a senior developer in our company. Implement the task below.
Follow the business rules and engineering guidelines from the internal knowledge strictly
and reuse the existing code base (configuration, errors, clock, repository).
Answer with the new or changed code only, followed by a short list of the rules you applied.

Task:
{task}

Internal knowledge:
{prepared_knowledge}

Code base:
{load_code_files(code_files)}
"""

    # TODO: Generate the code
    return generate_completion(
        user_prompt=prompt,
        system_prompt=SYSTEM_PROMPT,
        model_name=generation_model,
        temperature=0.2,
        max_output_tokens=2000,
    )

In [24]:
task = "Add a method extend_booking to the BookingService that extends a booking by a given number of minutes."
print(generate_code_with_rag(task, CODE_FILES, verbose=True))

Retrieved knowledge from: ['Document: domain/room-booking-policy.md', 'Document: domain/room-booking-policy.md', 'Document: domain/glossary.md', 'Document: domain/room-booking-policy.md', 'Document: security/data-protection.md'] 

```python
    def extend_booking(self, booking_id: str, minutes: int, user: User) -> Booking:
        """Extends an existing booking by a given number of minutes."""
        booking = self.get_booking(booking_id, user)
        
        if booking.owner_id != user.user_id and not user.is_admin:
            raise PermissionDeniedError("Only the owner or a facility admin can extend a booking.")
        
        if booking.status != BookingStatus.CONFIRMED and booking.status != BookingStatus.CHECKED_IN:
            raise PermissionDeniedError("Only active bookings can be extended.")

        new_end = booking.end + timedelta(minutes=minutes)
        duration_minutes = int((new_end - booking.start).total_seconds() / 60)

        # Validate duration
        if dura

Check the generated code:
* Does it use `self.clock()` instead of `datetime.now()` (PY-05)?
* Does it raise a `RoomieError` subclass instead of `HTTPException` (PY-06)?
* Does it check the maximum duration using `config.MAX_BOOKING_MINUTES` instead of a hard-coded `240` (PY-03)?
* Does it check for overlaps with the following booking?

For comparison, generate the code **without** the internal knowledge (only task and code). What is different?

In [22]:
# TODO: Generate the code without the internal knowledge and compare
code_only_prompt = f"""Implement the task below. Answer with the new or changed code only.

Task:
{task}

Code base:
{load_code_files(CODE_FILES)}
"""
print(generate_completion(user_prompt=code_only_prompt, model_name=generation_model, temperature=0.2, max_output_tokens=2000))

### File: roomie/services/booking_service.py

```python
    def extend_booking(self, booking_id: str, minutes: int, user: User) -> Booking:
        """Extends a booking by a given number of minutes."""
        booking = self.get_booking(booking_id, user)
        if booking.owner_id != user.user_id and not user.is_admin:
            raise PermissionDeniedError("Only the owner can extend a booking.")
        
        if booking.status != BookingStatus.CONFIRMED and booking.status != BookingStatus.CHECKED_IN:
            raise PermissionDeniedError("Only active bookings can be extended.")

        new_end = booking.end + timedelta(minutes=minutes)
        
        # Check for overlaps with other bookings
        existing = self.repository.list_bookings_for_room(booking.room_id)
        for other in existing:
            if other.booking_id == booking.booking_id or not other.is_active:
                continue
            # Check if the new end time overlaps with another booking's start ti

In the task above, the existing code already shows many conventions (the injected clock, the error classes), so the model might get them right even without the knowledge base. The knowledge base makes the biggest difference for rules that are **not visible in the code** at all. Try the following task with and without RAG: our security requirements (SEC-04) and data protection rules (DP-01) forbid logging meeting titles or names.

In [23]:
# TODO: Compare a task whose rules are only documented in the knowledge base
logging_task = "Add logging to BookingService.create_booking so that we can see in the logs who booked which meeting."
logging_files = ["roomie/services/booking_service.py", "roomie/notifications.py"]

print("========== WITH RAG ==========")
print(generate_code_with_rag(logging_task, logging_files, verbose=True))
print("========== WITHOUT RAG ==========")
print(generate_completion(
    user_prompt=f"Implement the task below. Answer with the new or changed code only.\n\nTask:\n{logging_task}\n\nCode base:\n{load_code_files(logging_files)}",
    model_name=generation_model,
    temperature=0.2,
    max_output_tokens=2000,
))

========== WITH RAG ==========
Retrieved knowledge from: ['Document: security/data-protection.md', 'Document: security/data-protection.md', 'Document: security/secrets-and-logging.md', 'Document: domain/room-booking-policy.md', 'Document: domain/room-booking-policy.md'] 

```python
import logging

# ... existing imports ...

logger = logging.getLogger("roomie.booking_service")

# ... inside BookingService class ...

    def create_booking(self, request: BookingCreate, user: User) -> Booking:
        room = self.repository.get_room(request.room_id)
        if room is None:
            raise RoomNotFoundError(f"Room {request.room_id} does not exist.")

        existing = self.repository.list_bookings_for_room(room.room_id)
        now = self.clock()
        validate_new_booking(request, room, existing, now)

        booking = Booking(
            booking_id=self.repository.new_booking_id(),
            room_id=room.room_id,
            owner_id=user.user_id,
            title=request.tit